In [8]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
from pathlib import Path
from gen_variable_standard_static import metrics_search_for_fragment_df

In [3]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records', 'sample_year'],
      dtype='str')

In [6]:
good_time_systems_copied = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]

In [23]:
systems_worthwhile = systems_cleaned[systems_cleaned['system_id'].isin(good_time_systems_copied)]
systems_worthwhile[['pvcz_composite', 'pvcz_t_rack', 'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind']]

,pvcz_composite,pvcz_t_rack,pvcz_t_roof,pvcz_humidity,pvcz_wind
2,12,2,4,1,3
3,12,2,4,1,3
4,12,2,4,1,3
7,12,2,4,1,3
8,12,2,4,1,3
9,12,2,4,1,3
10,34,4,7,3,3
15,34,4,6,3,3
68,12,2,4,1,3
69,12,2,4,1,3


In [5]:
temp_year = 1990

In [9]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [11]:
metrics_rel_systems = metrics_df[metrics_df['system_id'].isin(good_time_systems_copied)]
my_temp_metrics = metrics_search_for_fragment_df(metrics_rel_systems, 'temp')

In [12]:
my_temp_metrics['system_id'].unique()

array([  10, 1199, 1204, 1283, 1284, 1289,   33,   36, 4902, 4903,    4,
         50,   51], dtype=int32)

Syt

In [15]:
for system_id in good_time_systems_copied:
    print(f'System {system_id}')
    print(my_temp_metrics[my_temp_metrics['system_id'] == system_id][['metric_id', 'sensor_name', 'common_name']])

System 4
      metric_id    sensor_name           common_name
1719        320   ambient_temp   Temperature ambient
1720        321  module_temp_1    Temperature module
1721        322  module_temp_2    Temperature module
1722        323  module_temp_3    Temperature module
1723        324  inverter_temp  Temperature inverter
1725        325       das_temp     Temperature panel
System 10
    metric_id    sensor_name           common_name
7         428   ambient_temp   Temperature ambient
8         429  module_temp_1    Temperature module
9         430  module_temp_2    Temperature module
10        431  module_temp_3    Temperature module
11        432  inverter_temp  Temperature inverter
13        433       das_temp     Temperature panel
System 33
      metric_id    sensor_name           common_name
1311        589   ambient_temp   Temperature ambient
1312        590  module_temp_1    Temperature module
1313        591  module_temp_2    Temperature module
1314        592  module_temp_3 

In [22]:
# check for reality
for system_id in good_time_systems_copied:
    print(f'System {system_id}')
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_ind = relevant_rows_systems.index[0]
    my_sys_temp_metrics = my_temp_metrics[my_temp_metrics['system_id'] == system_id]
    my_metrics = set(my_sys_temp_metrics['metric_id'].unique())
    good_metrics = {
        metric_id: False for metric_id in my_metrics
    }
    for metric_id in my_metrics:
        data_read_pq = pq.ParquetDataset(
            f'../../../../data_ds_project/systems/parquet/{system_id}/',
            filters=[('metric_id', '==', metric_id)]
        )
        data_read_df = data_read_pq.read().to_pandas()
        if data_read_df.shape[0] > 100:
            good_metrics[metric_id] = True
    print(good_metrics)

System 4
{np.int32(320): True, np.int32(321): True, np.int32(322): True, np.int32(323): True, np.int32(324): True, np.int32(325): True}
System 10
{np.int32(428): True, np.int32(429): True, np.int32(430): True, np.int32(431): True, np.int32(432): True, np.int32(433): True}
System 33
{np.int32(589): True, np.int32(590): True, np.int32(591): True, np.int32(592): True, np.int32(593): True, np.int32(594): True}
System 36
{np.int32(640): True, np.int32(641): True, np.int32(659): True, np.int32(660): True, np.int32(661): True, np.int32(662): True, np.int32(634): True, np.int32(635): True, np.int32(636): True, np.int32(637): True, np.int32(638): True, np.int32(639): True}
System 50
{np.int32(770): True, np.int32(759): True, np.int32(760): True, np.int32(761): True, np.int32(762): True, np.int32(763): True, np.int32(764): True}
System 51
{np.int32(780): True, np.int32(781): True, np.int32(782): True, np.int32(783): True, np.int32(784): True, np.int32(785): True, np.int32(791): True}
System 1199